In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

df = pd.read_csv("final_cleaned_data.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1213 entries, 0 to 1212
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   company_names        1213 non-null   object 
 1   cars_names           1213 non-null   object 
 2   engines              1213 non-null   object 
 3   cc_battery_capacity  1213 non-null   object 
 4   horsepower           1213 non-null   float64
 5   total_speed          1213 non-null   float64
 6   cars_prices          1213 non-null   float64
 7   fuel_types           1213 non-null   object 
 8   seats                1213 non-null   object 
 9   torque               1213 non-null   float64
 10  engine_cc            1213 non-null   float64
 11  battery_kwh          1213 non-null   float64
 12  has_battery          1213 non-null   int64  
 13  has_engine           1213 non-null   int64  
 14  vehicle_type         1213 non-null   object 
 15  acceleration_0_100   1213 non-null   f

In [2]:
# Train/Test split
df["log_price"] = np.log1p(df["cars_prices"])
df["log_price"]

y = df["log_price"]
X = df.drop(columns=["log_price", "cars_prices"])


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
numeric_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

In [3]:
# Baseline Model
baseline_prediction = np.full(
    len(y_test),
    y_train.median()
)
baseline_mae = mean_absolute_error(
    y_test,
    baseline_prediction
)

baseline_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        baseline_prediction
    )
)

baseline_r2 = r2_score(
    y_test,
    baseline_prediction
)

print("Baseline MAE:", baseline_mae)
print("Baseline RMSE:", baseline_rmse)
print("Baseline R²:", baseline_r2)

Baseline MAE: 0.6924585375302733
Baseline RMSE: 0.9812017392757955
Baseline R²: -0.06289437235538453


In [4]:
# Model 1: Linear Regression
from sklearn.linear_model import LinearRegression

lr_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_mae = mean_absolute_error(y_test, lr_pred)

lr_rmse = np.sqrt(
    mean_squared_error(y_test, lr_pred)
)

lr_r2 = r2_score(y_test, lr_pred)

print("LinearRegressionModel MAE:", lr_mae)
print("LinearRegressionModel RMSE:", lr_rmse)
print("LinearRegressionModel R²:", lr_r2) 

LinearRegressionModel MAE: 0.18582385834188145
LinearRegressionModel RMSE: 0.2747456027975654
LinearRegressionModel R²: 0.9166635512493139


In [5]:
# Model 2: RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)

rf_rmse = np.sqrt(
    mean_squared_error(y_test, rf_pred)
)

rf_r2 = r2_score(y_test, rf_pred)
print("RandomForestRegressionModel MAE:", rf_mae)
print("RandomForestRegressionModel RMSE:", rf_rmse)
print("RandomForestRegressionModel R²:", rf_r2) 

RandomForestRegressionModel MAE: 0.1512044092376501
RandomForestRegressionModel RMSE: 0.21246058527714898
RandomForestRegressionModel R²: 0.950165485004328


In [ ]:
# Model 3: GradientBoostingRegressor
from sklearn.ensemble import GradientBoostingRegressor

gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        random_state=42
    ))
])

gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_pred)

gb_rmse = np.sqrt(
    mean_squared_error(y_test, gb_pred)
)

gb_r2 = r2_score(y_test, gb_pred)
print("GradientBoostingRegressorModel MAE:", gb_mae)
print("GradientBoostingRegressorModel RMSE:", gb_rmse)
print("GradientBoostingRegressorModel R²:", gb_r2) 

0.1798701925813878 0.2486491936518879 0.9317429366888996
GradientBoostingRegressorModel MAE: 0.1798701925813878
GradientBoostingRegressorModel RMSE: 0.2486491936518879
GradientBoostingRegressorModel R²: 0.9317429366888996


In [7]:
# Results
results = pd.DataFrame({
    "Model": [
        "Baseline",
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        baseline_mae,
        lr_mae,
        rf_mae,
        gb_mae
    ],
    "RMSE": [
        baseline_rmse,
        lr_rmse,
        rf_rmse,
        gb_rmse
    ],
    "R2": [
        baseline_r2,
        lr_r2,
        rf_r2,
        gb_r2
    ]
})

results.sort_values("RMSE") 

,Model,MAE,RMSE,R2
2,Random Forest,0.151204,0.212461,0.950165
3,Gradient Boosting,0.179870,0.248649,0.931743
1,Linear Regression,0.185824,0.274746,0.916664
0,Baseline,0.692459,0.981202,-0.062894
